In [6]:
from langchain_core.documents import Document

from langchain_community.document_loaders import (
    PyPDFLoader,
    JSONLoader,
    TextLoader
)

import os
import json

In [14]:
def load_json(file_path):

    with open(file_path) as f:
        data = json.load(f)

    content = data.get("description", "")

    doc = Document(
        page_content=content,
        metadata={
            "source": file_path,
            "type": "json",
            "title": data.get("title", "")
        }
    )

    return [doc]

In [13]:
def load_markdown(file_path):

    loader = TextLoader(file_path)

    docs = loader.load()

    for d in docs:
        d.metadata["type"] = "markdown"
        d.metadata["source"] = file_path

    return docs

In [12]:
def load_pdf(file_path):

    loader = PyPDFLoader(file_path)

    docs = loader.load()

    for d in docs:
        d.metadata["type"] = "pdf"
        d.metadata["source"] = file_path

    return docs

In [16]:
def validate_documents(docs):

    valid_docs = []

    for doc in docs:

        if not doc.page_content:
            continue

        if len(doc.page_content) < 20:
            continue

        valid_docs.append(doc)

    return valid_docs

In [11]:
def enrich_metadata(docs):

    for doc in docs:

        text = doc.page_content.lower()

        if "revenue" in text:
            doc.metadata["topic"] = "finance"

        elif "product" in text:
            doc.metadata["topic"] = "product"

        else:
            doc.metadata["topic"] = "general"

        doc.metadata["length"] = len(doc.page_content)

    return docs

In [10]:
def ingest_documents(folder):

    all_docs = []

    for file in os.listdir(folder):

        path = os.path.join(folder, file)

        if file.endswith(".json"):
            docs = load_json(path)

        elif file.endswith(".md"):
            docs = load_markdown(path)

        elif file.endswith(".pdf"):
            docs = load_pdf(path)

        else:
            continue

        all_docs.extend(docs)

    return all_docs

In [9]:
def context_pipeline(data_folder):

    docs = ingest_documents(data_folder)

    docs = validate_documents(docs)

    docs = enrich_metadata(docs)

    return docs

In [18]:
documents = context_pipeline("./data")

for doc in documents:
    print(doc)

page_content='Acme Technologies Inc.
Q4 2025 Business Performance Summary
October 1 – December 31, 2025  ·  Confidential
Executive Summary
Q4 2025 marked a strong close to the fiscal year for Acme Technologies. Total revenue reached
$4.2M, representing a 23% year-over-year increase and exceeding our internal target of $3.9M by
7.7%. Recurring revenue as a share of total revenue grew to 68%, up from 61% in Q4 2024, reflecting
the continued success of our subscription-first go-to-market strategy.
Customer acquisition accelerated in the quarter, with 87 new logos added across the Enterprise and
Mid-Market segments. Net Revenue Retention (NRR) held at 112%, driven by strong upsell
performance in the Platform tier. Gross margin improved to 74.1%, a 2.3 percentage point gain versus
Q3 2025, primarily due to infrastructure optimizations completed in October.
Operating expenses increased 11% quarter-over-quarter, largely attributable to planned headcount
expansion in Sales and Customer Success